# Dataset setup and preprocessing


In [ ]:
!pip install opencv-python numpy matplotlib imgaug

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
project_path = "/content/drive/MyDrive/Colab Notebooks/ComputerVision/semseter-project/"

## raw data

In [ ]:
dataset_path = project_path + "dataset/Dataset"

## extracting frames from video

In [ ]:
# folder to contain extracted frames
frames_path = project_path + "frames"

In [ ]:
!rm -rf "{frames_path}"

In [ ]:
os.makedirs(frames_path, exist_ok=True)

# processing each video
videos = [f for f in os.listdir(dataset_path) if f.endswith(('.mp4'))]

for video_file in videos:
    video_path = os.path.join(dataset_path, video_file)
    video_name = os.path.splitext(video_file)[0]

    # create subfolder
    video_output = os.path.join(frames_path, video_name)
    os.makedirs(video_output, exist_ok=True)

    # open video
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_interval = max(fps // 2, 1) # producing many frames per video

    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_interval == 0:
            output_path = os.path.join(video_output, f"frame_{saved_count:04d}.jpg")
            cv2.imwrite(output_path, frame)
            saved_count += 1

        frame_count += 1

    cap.release()
    print(f"{video_name}: extracted {saved_count} frames")

print("\nall videos processed!")

VID_20260408_173112: extracted 42 frames
VID_20260408_171903: extracted 40 frames
VID_20260408_173854: extracted 39 frames
VID_20260408_173310: extracted 68 frames
VID_20260408_172918: extracted 57 frames
VID_20260408_172556: extracted 41 frames
VID_20260408_174323: extracted 69 frames
VID_20260408_174147: extracted 28 frames
VID_20260408_171454: extracted 37 frames
VID_20260408_174235: extracted 53 frames
VID_20260408_174106: extracted 55 frames
VID_20260408_171126: extracted 36 frames
VID_20260408_173405: extracted 75 frames
VID_20260408_174524: extracted 85 frames


## preprocessing all extracted frames

In [ ]:
preprocessed_path = project_path + "preprocessed"

os.makedirs(preprocessed_path, exist_ok=True)

preprocessed_path

In [ ]:
# clear this folder
!rm -rf "{preprocessed_path}/*"

In [ ]:
import glob

all_frames_paths = list(glob.glob(frames_path + '/**/*.jpg', recursive=True))

for f in all_frames_paths[0:5]:
  print(f.replace(frames_path, preprocessed_path))

there are very many images, and its necessary to discard images of low importance or noisy or blurry or unclear

to solve this manual process, 2 functions are created:
- vegetation/greenness detection(check if a frame contains enough vegetation above some threshold value)
- blurriness/sharpness detection(check if a frame is blurry or unclear)



### greenness cover detector

In [ ]:
def has_vegetation(image_bgr, green_threshold=0.50):
    """
    Returns True if image contains vegetation (greenness) above threshold.

    Parameters:
    - image_bgr: input BGR image (OpenCV format)
    - green_threshold: fraction of pixels that must be green (0.0 - 1.0)

    Returns:
    - True/False (or dict if debug=True)
    """

    # convert to HSV color space
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

    # green color range in HSV
    lower_green = np.array([25, 40, 40])
    upper_green = np.array([95, 255, 255])

    # mask for green regions
    mask = cv2.inRange(hsv, lower_green, upper_green)

    # count green pixels
    green_pixels = np.count_nonzero(mask)

    # computing total pixels
    total_pixels = mask.shape[0] * mask.shape[1]

    # ratio of vegetation
    vegetation_ratio = green_pixels / total_pixels

    # decision
    result = vegetation_ratio >= green_threshold

    return result

### blurriness/sharpness detector

In [ ]:
def is_blurry(gray_image, threshold=80.0):
    """
    Returns True if image is blurry (unclear), False otherwise.

    Method:
    - Uses Variance of Laplacian (focus measure)

    Parameters:
    - image: input gray image
    - threshold: lower = stricter blur detection

    Returns:
    - True/False or dict (if debug=True)
    """

    # compute Laplacian (edge response)
    laplacian = cv2.Laplacian(gray_image, cv2.CV_64F)

    # variance of Laplacian = sharpness score
    variance = laplacian.var()

    # Decision rule
    blurry = variance < threshold

    return blurry

### preprocessing all images

In [ ]:
# counter
i = 0
no_vegetation_count = 0
blurry_count = 0

for image_path in all_frames_paths:
    i += 1 # track images processed

    image = cv2.imread(image_path)

    if image is None:
        print(f"error loading image: {image_path}")
        continue
    # print(f"processing {image_path}")


    # resizing image for consistency
    image = cv2.resize(image, (224,224))

    # filter: check for vegetation
    # discard image basing on vegation/greenness cover - taking 50% green frames
    if not has_vegetation(image, green_threshold=0.5):
        no_vegetation_count += 1
        display(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        print(f"discard: little vegation: {image_path}")
        continue

    # converting image to grayscale
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # filter: check for blurriness - discarding frames that are 50% blurry
    if is_blurry(gray, threshold=50.0):
        blurry_count += 1
        display(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        print(f"discard: blurry: {image_path}")
        continue

    # enhance contrast
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    enhanced = clahe.apply(gray)

    save_path = image_path.replace(frames_path, preprocessed_path)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    cv2.imwrite(save_path, enhanced)
    print(f"saved: {save_path}")

    if i % 20 == 0:
        plt.figure(figsize=(10,10))
        plt.subplot(1,2,1)
        plt.imshow(gray, cmap='gray')
        plt.subplot(1,2,2)
        plt.imshow(enhanced, cmap='gray')
        plt.show()
        print(f"{i}/{len(all_frames_paths)}")

print("\n\npreprocessing complete.")
print(f"\n\nno vegetation: {no_vegetation_count}")
print(f"blurry: {blurry_count}")

Output hidden; open in https://colab.research.google.com to view.

## cleanup

removing extracted frames to save storage space

In [ ]:
!rm -rf "{frames_path}"

## next steps

1. data annotation
2. dataset splitting
3. feature extraction + augmentation
4. model training and evaluation